# Kernels, SVMs and the nullspace

Notes and code from this class: the nullspace and why optimal weights are a combination of the data points,
feature maps (phi) for kernels, the SVC and RBF-kernel equations, a kernel SVM on the banknote data,
and an SVM / SVR on the Yosemite temperatures.

## 1. Nullspace and linear combinations

**What the nullspace of X is, and why it can be ignored.** The nullspace of X is the set of vectors v where X v = 0.
These are directions that every data point is perpendicular to, so the data has no information about them.
Adding such a v to the weights does not change any prediction on the data, because X(w + v) = Xw + Xv = Xw,
so the loss does not change either. The data cannot tell w from w + v, so we ignore those directions and use the
weights with no nullspace part (the smallest ones, which is what ridge picks automatically).

**Why the optimal weights are a linear combination of the data points.** Split any weights into a part that lies in the
span of the data points and a part in the nullspace. The loss only sees the span part, and the penalty only grows with
the nullspace part, so the best choice sets the nullspace part to zero. What is left is w = a1*x1 + a2*x2 + ... + aN*xN,
a linear combination of the data points. This holds whenever the loss depends on w only through its dot products with the
data points and the penalty grows with the size of w. It means predictions only need dot products between points,
which is what the kernel trick uses.

## 2. Feature maps (phi) for the kernels

```
Linear kernel:  k(x, z) = x . z
    phi(x) = x

Quadratic kernel:  k(x, z) = (x . z + 1)^2
    phi(x) = ( 1,
               sqrt(2)*x1, ..., sqrt(2)*xd,
               x1^2, ..., xd^2,
               sqrt(2)*xi*xj  for every pair i < j )
    2 features: (1, sqrt(2) x1, sqrt(2) x2, x1^2, x2^2, sqrt(2) x1 x2) -> 6 coordinates
    4 features: 15 coordinates

Plain-square version:  k(x, z) = (x . z)^2
    phi(x) = ( x1^2, ..., xd^2, sqrt(2)*xi*xj for every pair i < j )
    2 features: (x1^2, x2^2, sqrt(2) x1 x2) -> 3 coordinates;  4 features: 10 coordinates

Radial basis kernel: no finite phi (the feature space is infinite-dimensional).
```

The sqrt(2) is there so the dot product of two phi vectors gives the kernel: the cross term appears as sqrt(2) * sqrt(2) = 2.
So a kernel gives the dot product in the feature space without ever building it: (x . z)^2 is computed in the original space.

## 3. SVC and RBF kernel equations

```
Data: (x_i, y_i), i = 1..N, with y_i = -1 or +1. Predict the sign of  w . x + b.

Margin: the lines w . x + b = +1 and w . x + b = -1 are 2 / ||w|| apart.
Hard margin:  minimize (1/2) ||w||^2   subject to   y_i (w . x_i + b) >= 1

Soft margin (primal):
    minimize    (1/2) ||w||^2 + C * sum_i xi_i
    subject to  y_i (w . x_i + b) >= 1 - xi_i   and   xi_i >= 0
    xi_i = max(0, 1 - y_i f(x_i))  (hinge loss);  xi_i > 1 means the point is misclassified.
    Bigger C = less regularization = narrower margin.

Dual:
    w = sum_i alpha_i y_i x_i,   sum_i alpha_i y_i = 0,   0 <= alpha_i <= C
    maximize  sum_i alpha_i - (1/2) sum_i sum_j alpha_i alpha_j y_i y_j k(x_i, x_j)

What alpha_i says about point i:
    alpha_i = 0          outside the margin, correct side
    0 < alpha_i < C      exactly on the margin
    alpha_i = C          inside the margin or misclassified
    Support vectors are the points with alpha_i > 0.

Prediction:
    f(x) = sum_i alpha_i y_i k(x_i, x) + b ;  predict sign(f(x))
    b = y_k - sum_i alpha_i y_i k(x_i, x_k)  for any k with 0 < alpha_k < C

RBF kernel:
    k(x, z) = exp(-gamma ||x - z||^2) = exp(-||x - z||^2 / (2 sigma^2)),   gamma = 1 / (2 sigma^2)
    A similarity in (0, 1]: 1 when x = z, falling toward 0 with distance. Infinite-dimensional feature space.
    f(x) is a sum of RBF bumps centered at the support vectors, with heights alpha_i y_i.
    Tiny gamma: every pair looks alike, the boundary is nearly flat (underfits).
    Huge gamma: each point is similar only to itself, so it memorizes (overfits).
```

### Mistakes to look for when a model explains these

| A model may say | Correct |
|---|---|
| The margin is 1 / (length of w) | The two margin lines are 2 / (length of w) apart |
| Bigger C means more regularization, like ridge's alpha | Bigger C means less regularization and a narrower margin |
| The primal is maximized and the dual minimized | The primal is minimized and the dual is maximized |
| alpha_i <= C always applies | The cap C is for the soft margin; the hard-margin dual has no upper bound |
| Support vectors are the points on the margin | They also include points inside the margin and misclassified ones (alpha = C) |
| Compute b from any support vector | Only from points with 0 < alpha < C, which sit exactly on the margin |
| The sum of the slacks equals the number of mistakes | It is at least the number of mistakes |
| The dual drops y_i y_j, or adds the double sum | y_i y_j stays, the double sum has a minus sign, and sum alpha_i y_i = 0 stays |
| gamma = 1 / sigma^2, or gamma = sigma | gamma = 1 / (2 sigma^2) |
| Larger gamma gives a smoother boundary | Larger gamma means narrower bumps and overfitting |
| RBF measures distance | It measures similarity |
| RBF maps to a finite higher dimension | The feature space is infinite-dimensional |
| You can write w for the RBF model | There is no explicit w, only the sum over support vectors |
| Every training point appears in f(x) | Only support vectors do |

## 4. A simple SVC and an SVC with a radial basis kernel

`decision_by_hand` rebuilds f(x) = sum_i alpha_i y_i k(x_i, x) + b from the fitted model's own numbers,
to check the formula against scikit-learn.

In [5]:
import numpy as np
from sklearn.svm import SVC


def make_linear_svc(X, y, C=1.0):
    """Fit a soft-margin SVC with a straight-line (linear) decision boundary.

    C is the price of a point that lands inside the margin or on the wrong side:
    large C = costly, so the margin narrows and the fit hugs the training data;
    small C = cheap, so the margin widens and the boundary is smoother.
    Returns the fitted sklearn SVC.
    """
    return SVC(kernel="linear", C=C).fit(X, y)


def make_rbf_svc(X, y, C=1.0, gamma=None):
    """Fit a soft-margin SVC with the radial basis kernel k(x, z) = exp(-gamma * ||x - z||^2).

    gamma sets how far one training point's influence reaches: large gamma = narrow
    bumps (wiggly boundary, can overfit); small gamma = wide bumps (smooth, close to linear).
    If gamma is None we use sklearn's default rule 1 / (n_features * X.var()), but as an
    explicit number so we know the value that was used.
    Returns the fitted sklearn SVC.
    """
    X = np.asarray(X)
    if gamma is None:
        gamma = 1.0 / (X.shape[1] * X.var())
    return SVC(kernel="rbf", C=C, gamma=gamma).fit(X, y)


def decision_by_hand(model, X_new):
    """Rebuild f(x) = sum_i (alpha_i * y_i) * k(x_i, x) + b from the fitted model's own numbers.

    Only the support vectors (alpha_i > 0) appear in the sum. Used to check that the
    textbook dual formula matches what sklearn computes.
    """
    X_new = np.asarray(X_new)
    SV = model.support_vectors_
    coef = model.dual_coef_[0]               # alpha_i * y_i, one per support vector
    if model.kernel == "linear":
        K = X_new @ SV.T                     # k(x_i, x) = x_i . x
    else:
        sq_dist = ((X_new[:, None, :] - SV[None, :, :]) ** 2).sum(-1)
        K = np.exp(-model.gamma * sq_dist)   # k(x_i, x) = exp(-gamma ||x_i - x||^2)
    return K @ coef + model.intercept_[0]

In [6]:
from sklearn.datasets import make_circles
from sklearn.model_selection import train_test_split

# a ring around a blob: a straight line cannot separate it, the radial basis kernel can
X, y = make_circles(300, noise=0.1, factor=0.4, random_state=0)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0)
for name, maker in (("linear", make_linear_svc), ("rbf", make_rbf_svc)):
    m = maker(Xtr, ytr)
    gap = np.abs(decision_by_hand(m, Xte) - m.decision_function(Xte)).max()
    print(f"{name}: test accuracy {m.score(Xte, yte):.3f}, by-hand vs sklearn max gap {gap:.1e}")

linear: test accuracy 0.411, by-hand vs sklearn max gap 6.3e-15
rbf: test accuracy 1.000, by-hand vs sklearn max gap 2.2e-15


## 5. Kernel SVM on the banknote data

From the guide at stackabuse.com ("Implementing SVM and Kernel SVM with Python's Scikit-Learn"): the UCI banknote
authentication data, `test_size=0.20`, `random_state=42`. Part 1 is the guide's linear SVM; parts 2 and 3 add the
linear, quadratic and radial basis options and compare them with cross-validation.

The quadratic kernel is (x . z + 1)^2. The plain square (x . z)^2 gives the same score for x and -x
(f(x) = f(-x)), so it cannot separate classes that sit on opposite sides of the data's average; it is kept as an extra row.

In [7]:
# Load the data (as in the guide)
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.model_selection import train_test_split, cross_val_score, RepeatedStratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

data_link = "https://archive.ics.uci.edu/ml/machine-learning-databases/00267/data_banknote_authentication.txt"
col_names = ["variance", "skewness", "curtosis", "entropy", "class"]
bankdata = pd.read_csv(data_link, names=col_names, sep=",", header=None)
print(bankdata.shape)
print(bankdata["class"].value_counts())

X = bankdata.drop("class", axis=1)
y = bankdata["class"]
SEED = 42
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=SEED)


# Part 1: simple kernel SVM (the guide's linear SVM)
svc = SVC(kernel="linear")
svc.fit(X_train, y_train)
y_pred = svc.predict(X_test)
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))


# Part 2: options for linear, quadratic and radial basis
def kernel_svm(kernel="linear", C=1.0, gamma="scale", coef0=1.0, scale=True):
    """Build (not fit) an SVM with the chosen kernel.

    linear    : k(x, z) = x . z
    quadratic : k(x, z) = (x . z + coef0)^2. With coef0=1 this has constant, linear and squared
                terms. coef0=0 gives the plain square (x . z)^2, which scores x and -x the same
                and cannot separate classes that sit on opposite sides of the data's average.
                (sklearn's poly kernel is (gamma * x.z + coef0)^degree, so gamma=1 is needed.)
    rbf       : k(x, z) = exp(-gamma * ||x - z||^2)
    scale=True puts the features on a common scale first (SVMs are sensitive to this);
    scale=False matches the guide, which fits on the raw columns.
    """
    if kernel == "linear":
        svc = SVC(kernel="linear", C=C)
    elif kernel == "quadratic":
        svc = SVC(kernel="poly", degree=2, gamma=1.0, coef0=coef0, C=C)
    elif kernel == "rbf":
        svc = SVC(kernel="rbf", gamma=gamma, C=C)
    else:
        raise ValueError("kernel must be 'linear', 'quadratic' or 'rbf'")
    return make_pipeline(StandardScaler(), svc) if scale else svc


for k in ("linear", "quadratic", "rbf"):
    model = kernel_svm(k).fit(X_train, y_train)
    print(f"--- {k} ---")
    print(confusion_matrix(y_test, model.predict(X_test)))
    print(classification_report(y_test, model.predict(X_test)))


# Part 3: which kernel adds the most accuracy?
# The test set is small (275 notes), so one wrong note moves accuracy by about 0.4 points.
# Cross-validation (5 folds, repeated 3 times) gives a steadier comparison.
models = {
    "linear": kernel_svm("linear"),
    "quadratic": kernel_svm("quadratic"),
    "rbf": kernel_svm("rbf"),
    "quadratic, plain square (extra)": kernel_svm("quadratic", coef0=0.0),
}
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=SEED)
rows = []
for name, est in models.items():
    pred = clone(est).fit(X_train, y_train).predict(X_test)
    cv_scores = cross_val_score(clone(est), X, y, cv=cv)
    rows.append({"kernel": name,
                 "test_accuracy": accuracy_score(y_test, pred),
                 "test_errors": int((pred != y_test).sum()),
                 "cv_mean": cv_scores.mean(),
                 "cv_std": cv_scores.std()})
results = pd.DataFrame(rows).set_index("kernel")
results["cv_gain_over_linear"] = results["cv_mean"] - results.loc["linear", "cv_mean"]
print(results.sort_values("cv_mean", ascending=False).round(4).to_string())

(1372, 5)
class
0    762
1    610
Name: count, dtype: int64
[[146   2]
 [  2 125]]
              precision    recall  f1-score   support

           0       0.99      0.99      0.99       148
           1       0.98      0.98      0.98       127

    accuracy                           0.99       275
   macro avg       0.99      0.99      0.99       275
weighted avg       0.99      0.99      0.99       275

--- linear ---
[[146   2]
 [  2 125]]
              precision    recall  f1-score   support

           0       0.99      0.99      0.99       148
           1       0.98      0.98      0.98       127

    accuracy                           0.99       275
   macro avg       0.99      0.99      0.99       275
weighted avg       0.99      0.99      0.99       275

--- quadratic ---
[[148   0]
 [  0 127]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       148
           1       1.00      1.00      1.00       127

    accuracy        

## 6. SVM and SVR on the Yosemite temperatures

Run this after the first Yosemite cell (it needs `data` and `valid`). The model fits on 2011-2013, picks its settings on 2014
and is scored on 2015-2016, because neighbouring readings are nearly identical and a random split would leak.

- A: one temperature reading -> month.
- B: a day of 24 hourly temperatures -> month.
- C: SVR predicting temperature from the time of day and time of year, for an R^2 to compare with the linear parameter model.

In [8]:
# SVM and SVR on the Yosemite temperatures (run AFTER the first cell: needs `data` and `valid`)
import numpy as np
import pandas as pd
from sklearn.compose import TransformedTargetRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC, SVR

years = range(2011, 2017)
base = "https://www.ncei.noaa.gov/pub/data/uscrn/products/subhourly01/%d/CRNS0101-05-%d-CA_Yosemite_Village_12_W.txt"
files = [base % (y, y) for y in years]
usecols = [1, 2, 8]

data = [np.loadtxt(f, usecols=usecols) for f in files]
data = np.vstack(data)

# Map from HHmm to an integer
data[:, 1] = np.floor_divide(data[:, 1], 100) * 60 + np.mod(data[:, 1], 100)
valid = data[:, 2] > -1000

d = data[valid]
dates = pd.to_datetime(d[:, 0].astype(int).astype(str), format="%Y%m%d")
year, month = dates.year.values, dates.month.values
minute, temp = d[:, 1], d[:, 2]


def kernel_svm(kernel, C=1.0, gamma="scale"):
    """Scaled SVC with a linear, quadratic ((x . z + 1)^2) or radial basis kernel."""
    if kernel == "linear":
        svc = SVC(kernel="linear", C=C)
    elif kernel == "quadratic":
        svc = SVC(kernel="poly", degree=2, gamma=1.0, coef0=1.0, C=C)
    else:
        svc = SVC(kernel="rbf", gamma=gamma, C=C)
    return make_pipeline(StandardScaler(), svc)


def circular_gap(a, b):
    """Distance in months around the year, so December and January are 1 apart."""
    g = np.abs(a - b)
    return np.minimum(g, 12 - g)


def month_search(X, label, yr, n_fit=4000, seed=0):
    """Fit on 2011-2013, pick kernel / C / gamma on 2014, then score on the unseen years 2015-2016.

    Neighbouring readings are almost identical, so a random split would leak; splitting by
    year avoids that. n_fit caps the rows used to fit (SVMs get slow as rows grow).
    """
    rng = np.random.default_rng(seed)
    fit = np.flatnonzero(yr <= 2013)
    fit = rng.choice(fit, min(n_fit, len(fit)), replace=False)
    val, test = np.flatnonzero(yr == 2014), np.flatnonzero(yr >= 2015)
    val, test = val[::max(1, len(val) // 1500)], test[::max(1, len(test) // 3000)]
    grid = ([("linear", C, "scale") for C in (0.1, 1, 10)]
            + [("quadratic", C, "scale") for C in (0.1, 1, 10)]
            + [("rbf", C, g) for C in (1, 10, 100) for g in (0.1, 1, 10)])
    scored = []
    for kernel, C, g in grid:
        m = kernel_svm(kernel, C, g).fit(X[fit], label[fit])
        scored.append((m.score(X[val], label[val]), kernel, C, g))
    scored.sort(key=lambda t: t[0], reverse=True)
    print("best settings on 2014 (accuracy, kernel, C, gamma):")
    for row in scored[:3]:
        print("  ", tuple(round(v, 3) if isinstance(v, float) else v for v in row))
    _, kernel, C, g = scored[0]
    model = kernel_svm(kernel, C, g).fit(X[fit], label[fit])
    pred, truth = model.predict(X[test]), label[test]
    wrong = pred != truth
    print(f"unseen years 2015-2016: exact month {np.mean(pred == truth):.3f}, "
          f"within one month {np.mean(circular_gap(pred, truth) <= 1):.3f}, "
          f"always guessing one month {np.bincount(truth).max() / len(truth):.3f}")
    errors = pd.Series(list(zip(truth[wrong], pred[wrong]))).value_counts().head(5)
    print("most common mistakes (true month -> predicted month: count):",
          ", ".join(f"{int(a)}->{int(b)}: {n}" for (a, b), n in errors.items()))
    return model


# A. one temperature reading -> month
print("A. one temperature reading -> month")
model_a = month_search(temp.reshape(-1, 1), month, year)

# B. a whole day of raw temperatures (24 hourly means) -> month
hourly = pd.DataFrame({"date": dates, "hour": (minute // 60).astype(int), "temp": temp})
days = hourly.pivot_table(index="date", columns="hour", values="temp", aggfunc="mean").dropna()
print(f"\nB. a day of hourly temperatures -> month ({days.shape[0]} complete days)")
model_b = month_search(days.to_numpy(), days.index.month.values, days.index.year.values)


# C. SVR: predict the temperature itself from the time of day and time of year, and compare R^2
theta_day = 2 * np.pi * minute / 1440
theta_year = 2 * np.pi * (dates.dayofyear.values - 1 + minute / 1440) / 365.25
T = np.column_stack([np.cos(theta_day), np.sin(theta_day), np.cos(theta_year), np.sin(theta_year)])

rng = np.random.default_rng(0)
fit = rng.choice(np.flatnonzero(year <= 2013), 4000, replace=False)
val = np.flatnonzero(year == 2014)[::24]
test = np.flatnonzero(year >= 2015)[::12]


def make_svr(C, gamma, epsilon=0.1):
    """RBF-kernel SVR; the temperature is standardized so epsilon is in standard deviations."""
    return TransformedTargetRegressor(regressor=SVR(kernel="rbf", C=C, gamma=gamma, epsilon=epsilon),
                                      transformer=StandardScaler())


best = None
for C in (1, 10, 100):
    for gamma in (0.03, 0.1, 0.3, 1, 3):
        r2 = make_svr(C, gamma).fit(T[fit], temp[fit]).score(T[val], temp[val])
        if best is None or r2 > best[0]:
            best = (r2, C, gamma)
print(f"\nC. SVR: best on 2014: R^2 {best[0]:.3f} with C={best[1]}, gamma={best[2]}")
svr = make_svr(best[1], best[2]).fit(T[fit], temp[fit])
pred_test = svr.predict(T[test])
print(f"R^2 on the fitted years {svr.score(T[fit], temp[fit]):.3f}, on unseen years 2015-2016 {svr.score(T[test], temp[test]):.3f}, "
      f"RMSE {np.sqrt(np.mean((temp[test] - pred_test) ** 2)):.2f} C, "
      f"support vectors {len(svr.regressor_.support_)} of {len(fit)}")

A. one temperature reading -> month
best settings on 2014 (accuracy, kernel, C, gamma):
   (0.213, 'rbf', 1, 10)
   (0.211, 'rbf', 10, 10)
   (0.209, 'rbf', 10, 1)
unseen years 2015-2016: exact month 0.146, within one month 0.440, always guessing one month 0.085
most common mistakes (true month -> predicted month: count): 8->7: 133, 12->2: 108, 6->7: 102, 8->9: 97, 2->1: 96

B. a day of hourly temperatures -> month (2184 complete days)
best settings on 2014 (accuracy, kernel, C, gamma):
   (0.407, 'rbf', 100, 0.1)
   (0.385, 'quadratic', 10, 'scale')
   (0.382, 'rbf', 10, 0.1)
unseen years 2015-2016: exact month 0.356, within one month 0.670, always guessing one month 0.085
most common mistakes (true month -> predicted month: count): 6->7: 30, 5->6: 18, 4->5: 16, 1->3: 14, 3->10: 14

C. SVR: best on 2014: R^2 0.615 with C=10, gamma=0.1
R^2 on the fitted years 0.620, on unseen years 2015-2016 0.579, RMSE 4.98 C, support vectors 3451 of 4000
